# SAM 2 for Roman Space Telescope: Tutorial 03
## Training from Paired FITS Directories with Point Prompts

This tutorial demonstrates how to train/fine-tune SAM 2 directly from **two separate directories**:
1. **Images Directory**: `/Volumes/T7/storage/ROMAN/test/synthetic/original/*.fits`
2. **Masks Directory**: `/Volumes/T7/storage/ROMAN/test/synthetic/masks/*.fits`

### Key Features Demonstrated:
- **Directory Pairing**: Automatically matching images and masks by filename.
- **Point Prompts (`prompt_type='point'`)**: Training SAM 2 using target foreground clicks instead of bounding boxes.
- **On-the-Fly Object Cropping (`crop_size=512`)**: Extracting $512 \times 512$ patches around sources from full $4088 \times 4088$ Roman WFI detector images in 1ms using memory-mapped I/O, preserving native pixel resolution.
- **Fast Low-Epoch Training**: Fine-tuning the prompt encoder and mask decoder on Apple Silicon MPS with live validation tracking.

### Google Colab Setup (Optional)
If running this notebook in **Google Colab** with an **A100 GPU**:
1. Go to **Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU**.
2. Uncomment and execute the cell below to install `sam2roman` and optionally mount Google Drive.

In [2]:
# In Google Colab, uncomment and run:
!git clone https://github.com/PabloMSanAla/sam2roman.git
%cd sam2roman
!pip install -e .
#
# If your ROMAN data is stored on Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# images_dir = Path('/content/drive/MyDrive/ROMAN/test/synthetic/original')
# masks_dir  = Path('/content/drive/MyDrive/ROMAN/test/synthetic/masks')

Cloning into 'sam2roman'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 94 (delta 32), reused 93 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 4.13 MiB | 18.40 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/sam2roman
Obtaining file:///content/sam2roman
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sam2roman (pyproject.toml) ... done
  Created wheel for sam2roman: filename=sam2roman-0.1.0-0.editable-py3-none-any.whl size=8680 sha256=c114736cc64781e5c3438bd35b6fbfd0bb4ec4faa999846cd62b6f3b9d725533
  Stored in directory: /tmp/pip-ephem-wheel-cache-q5ef_r52/wheels/02/2d/8e/ee1b0659d47fd8b53635375fdbc9381148cfde036c4195175d
Successfully built sam2roman


### Step 1: Import Libraries and Define Directory Paths

In [1]:
import sys
from pathlib import Path

for p in ['/content/sam2roman/src', 'src', '../src', '../../src']:
    resolved = Path(p).resolve()
    if resolved.exists() and str(resolved) not in sys.path:
        sys.path.insert(0, str(resolved))
        break

import matplotlib.pyplot as plt
import numpy as np
import torch

import sam2roman
from sam2roman.training.dataset import AstronomicalDataset
from sam2roman.training.trainer import SAM2Trainer
from sam2roman.inference.segmentor import SAM2Segmentor

print(f"sam2roman version: {sam2roman.__version__}")
print(f"PyTorch version:   {torch.__version__}")
print(f"Device:            {'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'}")

# Paths to your Roman synthetic data
images_dir = Path("/Volumes/T7/storage/ROMAN/test/synthetic/original")
masks_dir = Path("/Volumes/T7/storage/ROMAN/test/synthetic/masks")

if not images_dir.exists() or not masks_dir.exists():
    print("Warning: External paths not found. Using local sample_data fallback.")
    images_dir = Path("sample_data")
    masks_dir = Path("sample_data")

W1025 17:37:29.471000 88803 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


sam2roman version: 0.1.0
PyTorch version:   2.14.1
Device:            mps


### Step 2: Create Dataset from Paired Directories
We use `AstronomicalDataset.from_directories()`. It pairs matching `.fits` files and sets up on-the-fly $512 \times 512$ cutout sampling around sources with point prompts.

In [ ]:
dataset = AstronomicalDataset.from_directories(
    images_dir=images_dir,
    masks_dir=masks_dir,
    pattern="*.fits",
    crop_size=512,            # Extracts 512x512 windows around objects from 4088x4088 exposures
    prompt_type="point",       # Use POINT prompts (clicks on foreground)
    samples_per_file=None,     # None = ALL patches possible (every object across all files!)
    stretch_type="asinh",
    max_samples=4,             # Number of FITS files to match (or None for all 108 files)
)
print(f"Extracted ALL {len(dataset)} possible patches across the files!")

# Train (80%) / Validation (20%) split
n_train = max(1, int(0.8 * len(dataset)))
n_val = len(dataset) - n_train
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

Matched 16 image-mask pairs from directories!
Train samples: 12 | Val samples: 4


### Step 3: Inspect a Sample Cutout & Point Prompt
Notice how the point prompt is positioned exactly on the target object in local crop coordinates.

In [ ]:
sample = train_ds[0]
img_pil = sample["pil_image"]
mask_np = sample["mask"][0].numpy()
pt = sample["point"][0].tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.imshow(img_pil)
ax1.plot(pt[0], pt[1], 'r*', markersize=14, label=f"Point Prompt ({pt[0]:.0f}, {pt[1]:.0f})")
ax1.set_title("Astronomical Cutout (asinh stretch)")
ax1.legend()

ax2.imshow(mask_np, cmap="gray")
ax2.plot(pt[0], pt[1], 'r*', markersize=14)
ax2.set_title(f"Ground Truth Target Mask (Area: {int(mask_np.sum())} px)")

plt.tight_layout()
plt.show()

### Step 4: Configure `SAM2Trainer`
We freeze the vision encoder and configure point prompt training.

In [ ]:
trainer = SAM2Trainer(
    model_id="facebook/sam2.1-hiera-tiny",
    learning_rate=1e-4,
    weight_decay=1e-4,
    freeze_vision_encoder=True,
    prompt_type="point",  # Training with point prompts
)

print(f"Device:               {trainer.device}")
print(f"Trainable parameters: {sum(p.numel() for p in trainer.trainable_params):,}")

### Step 5: Execute Training Loop (Small Epochs Test)
We train for 3 epochs with live validation metrics.

In [ ]:
output_dir = Path("checkpoints")
output_dir.mkdir(parents=True, exist_ok=True)

history = trainer.train(
    dataset=train_ds,
    val_dataset=val_ds,
    epochs=3,
    batch_size=2,
    output_dir=output_dir,
    verbose=True,
)

### Step 6: Plot Loss Convergence and Validation IoU

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs, history["train_loss"], 'o-', label="Train Loss (BCE + Dice)", color="royalblue")
if history["val_loss"]:
    ax1.plot(epochs, history["val_loss"], 's--', label="Validation Loss", color="crimson")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Convergence")
ax1.grid(True, alpha=0.3)
ax1.legend()

if history["val_iou"]:
    ax2.plot(epochs, history["val_iou"], '^-', label="Validation Mean IoU", color="forestgreen")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Mean IoU")
    ax2.set_title("Validation IoU")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

plt.tight_layout()
plt.show()

### Step 7: Test Inference on a Roman Validation Patch with Fine-Tuned Weights

In [ ]:
best_checkpoint = output_dir / "checkpoint_best.pt"
if not best_checkpoint.exists():
    best_checkpoint = output_dir / "checkpoint_epoch_3.pt"

# Load fine-tuned weights into segmentor
segmentor = SAM2Segmentor(model_id="facebook/sam2.1-hiera-tiny")
segmentor.model_wrapper.model.load_state_dict(
    torch.load(best_checkpoint, map_location=trainer.device)["model_state_dict"]
)
segmentor.model_wrapper.model.eval()
print(f"Loaded fine-tuned model weights from: {best_checkpoint}")

val_sample = val_ds[0]
test_patch = val_sample["raw_data"]
target_pt = val_sample["point"][0].tolist()

res = segmentor.segment_points(
    image=test_patch,
    points=[(target_pt[0], target_pt[1])],
    stretch_type="asinh",
    verbose=True,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(val_sample["pil_image"])
ax1.plot(target_pt[0], target_pt[1], 'r*', markersize=14, label="Input Click Point")
ax1.set_title("Test Patch + Prompt Point")
ax1.legend()

ax2.imshow(val_sample["pil_image"])
if res.masks:
    pred_mask = res.masks[0]
    overlay = np.zeros((*pred_mask.shape, 4))
    overlay[pred_mask] = [0.2, 0.8, 0.2, 0.6]  # Green mask
    ax2.imshow(overlay)
    ax2.set_title(f"Predicted Mask (Area: {res.areas[0]} px, Score: {res.scores[0]:.2f})")

plt.tight_layout()
plt.show()